<a href="https://colab.research.google.com/github/marcocslima/dev/blob/master/Extra%C3%A7%C3%A3o_Notas_GISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#SETUP

In [ ]:
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver
!pip install selenium
# Limpa navegadores corrompidos
!apt-get purge -y google-chrome-stable chromium-browser chromium-chromedriver
!rm -rf /usr/bin/chromedriver /usr/local/bin/chromedriver

# Baixa e instala o Google Chrome Oficial
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!apt-get -f install -y

# Atualiza o Selenium
!pip install selenium --upgrade

In [ ]:
import os
import glob
import pandas as pd
from bs4 import BeautifulSoup

# 1. Defina o caminho onde seus arquivos XML estão armazenados no Colab.
# Se você fez upload direto para o Colab, use '/content/*.xml'
# Se estiver em uma pasta específica, mude para '/content/sua_pasta/*.xml'
caminho_arquivos = '/content/drive/MyDrive/tmp/XMLs Extracao/extraidos/*.xml'

# Lista (array) que vai armazenar os dicionários com os dados de cada nota
dados_extraidos = []

#Carregar pela base XML

In [ ]:
# 2. Encontra todos os arquivos XML no diretório especificado
arquivos_xml = glob.glob(caminho_arquivos)

if not arquivos_xml:
    print("Nenhum arquivo XML encontrado. Verifique se você fez o upload para a pasta correta.")
else:
    print(f"{len(arquivos_xml)} arquivo(s) encontrado(s). Iniciando extração...\n")

    # 3. Iterar sobre cada arquivo XML encontrado
    for arquivo in arquivos_xml:
        try:
            # Abrir e ler o conteúdo do arquivo
            with open(arquivo, 'r', encoding='utf-8') as f:
                conteudo_xml = f.read()

            # Passar o conteúdo para o BeautifulSoup fazer o parse (leitura)
            soup = BeautifulSoup(conteudo_xml, 'xml')

            # Encontrar a tag <InfNfse> para garantir que pegamos as informações certas
            inf_nfse = soup.find('InfNfse')

            if inf_nfse:
                # Extrair o Número e o Código de Verificação
                numero = inf_nfse.find('Numero').text if inf_nfse.find('Numero') else None
                codigo_verif = inf_nfse.find('CodigoVerificacao').text if inf_nfse.find('CodigoVerificacao') else None

                # Adicionar os dados extraídos no array
                dados_extraidos.append({
                    'Nome_do_Arquivo': os.path.basename(arquivo),
                    'Numero_NFSe': numero,
                    'Codigo_Verificacao': codigo_verif
                })
            else:
                print(f"Tag <InfNfse> não encontrada no arquivo: {arquivo}")

        except Exception as e:
            print(f"Erro ao processar o arquivo {arquivo}: {e}")

    # 4. Converter o array (lista de dicionários) em uma Tabela (DataFrame) do Pandas
    df_tabela = pd.DataFrame(dados_extraidos)
    df_tabela = df_tabela.sort_values(by='Numero_NFSe')

    # 5. Exibir a tabela na tela do Colab
    display(df_tabela)

    # (Opcional) Salvar essa tabela em um arquivo Excel ou CSV, caso precise
    # df_tabela.to_excel('/content/notas_extraidas.xlsx', index=False)
    # df_tabela.to_csv('/content/notas_extraidas.csv', index=False)

In [ ]:
df_tabela.columns

#Carregar pela base Planilha (DPOs)

In [ ]:
import pandas as pd
import re
import unicodedata

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = texto.replace('º', ' ').replace('ª', ' ').replace('°', ' ')
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[^\w\s]', ' ', texto)
    return ' '.join(texto.split())

def apenas_numeros(texto):
    if pd.isna(texto):
        return ""

    texto_str = str(texto).strip()

    if texto_str.endswith('.0'):
        texto_str = texto_str[:-2]

    return re.sub(r'\D', '', texto_str)

def buscar_notas_por_tomador(caminho_planilha, documento_alvo):
    # Lê tudo como texto para preservar os zeros à esquerda e evitar notação científica
    if caminho_planilha.endswith('.csv'):
        df = pd.read_csv(caminho_planilha, sep=None, engine='python', dtype=str)
    else:
        df = pd.read_excel(caminho_planilha, dtype=str)

    # Dicionário atualizado: Agora incluímos o Prestador
    palavras_chave = {
        'nota':[
            'n de nfs e', 'nr nota', 'numero da nota', 'numero da nfs e',
            'numero nota', 'nro nota', 'numero nfs', 'n nfs e', 'n nfs'
        ],
        'codigo':[
            'codigo de verificacao', 'cd verificacao', 'cod verificacao', 'verificador'
        ],
        'tomador':[
            'tomador cnpj', 'cnpj tomador', 'cpf tomador', 'tomador cpf',
            'doc tomador', 'documento tomador', 'cpf cnpj tomador'
        ],
        'prestador':[
            'prestador cnpj', 'cnpj prestador', 'cpf prestador', 'prestador cpf',
            'doc prestador', 'documento prestador', 'cpf cnpj prestador'
        ]
    }

    # Adicionado o 'prestador'
    colunas_mapeadas = {'nota': None, 'codigo': None, 'tomador': None, 'prestador': None}

    colunas_originais = df.columns
    for col_original in colunas_originais:
        col_norm = normalizar_texto(col_original)

        for tipo, chaves in palavras_chave.items():
            if any(chave in col_norm for chave in chaves):
                if colunas_mapeadas[tipo] is None:
                    colunas_mapeadas[tipo] = col_original

    faltando =[tipo for tipo, col in colunas_mapeadas.items() if col is None]
    if faltando:
        raise ValueError(f"Não consegui identificar as colunas para: {faltando}.")

    print(f"✓ Colunas identificadas com sucesso!")
    print(f"  - Nota: '{colunas_mapeadas['nota']}'")
    print(f"  - Código: '{colunas_mapeadas['codigo']}'")
    print(f"  - Tomador: '{colunas_mapeadas['tomador']}'")
    print(f"  - Prestador: '{colunas_mapeadas['prestador']}'\n")

    # Limpa o documento alvo e a coluna da planilha
    doc_alvo_limpo = apenas_numeros(documento_alvo)
    df['doc_tomador_limpo'] = df[colunas_mapeadas['tomador']].apply(apenas_numeros)

    # Filtra
    df_filtrado = df[df['doc_tomador_limpo'] == doc_alvo_limpo].copy()

    if df_filtrado.empty:
        print(f"Nenhuma nota encontrada para o CNPJ/CPF: {documento_alvo}")
        return pd.DataFrame()

    print(f"★ Foram encontradas {len(df_filtrado)} notas para este Tomador!")

    # Monta a tabela final agora incluindo o CNPJ do Prestador
    df_tabela = pd.DataFrame({
        'Numero_NFSe': df_filtrado[colunas_mapeadas['nota']].apply(lambda x: str(x).replace('.0', '') if str(x).endswith('.0') else x),
        'Codigo_Verificacao': df_filtrado[colunas_mapeadas['codigo']],
        'CNPJ_Prestador': df_filtrado[colunas_mapeadas['prestador']].apply(apenas_numeros) # Deixa só os números
    })

    df_tabela.dropna(subset=['Numero_NFSe', 'Codigo_Verificacao', 'CNPJ_Prestador'], inplace=True)
    df_tabela.reset_index(drop=True, inplace=True)

    return df_tabela

In [ ]:
# =================================================================
# NOTAS DPO:
# =================================================================

# 1. Defina o caminho da sua planilha no Colab
caminho_da_sua_planilha = '/content/drive/MyDrive/tmp/Extrações Notas DPOs/2 extração -05-2023_08-2025.xlsx' # Altere para o nome do arquivo que você fez upload

# 2. Defina o CNPJ ou CPF do Tomador que você quer buscar (Pode colocar com ou sem pontuação)
cnpj_tomador_busca = '49.850.425/0001-89' # Substitua pelo CNPJ real que deseja buscar

try:
    # 3. Executa a função
    df_tabela = buscar_notas_por_tomador(caminho_da_sua_planilha, cnpj_tomador_busca)

    # Exibe o resultado pronto para ir pro robô de Selenium
    display(df_tabela)

except FileNotFoundError:
    print(f"Arquivo não encontrado. Faça o upload do arquivo '{caminho_da_sua_planilha}' no Colab.")
except Exception as e:
    print(f"Erro: {e}")

#Rodar Extração

#Extração JSON

In [ ]:
# ==============================================================
# CÉLULA INTEGRADA: CARREGAMENTO DO FILTRO DE NFS-E DO CONSULTOR
# ==============================================================
import json
from google.colab import files

print("Selecione o arquivo JSON de notas baixado do aplicativo:")
uploaded = files.upload()

if uploaded:
    # Captura o arquivo JSON enviado pelo usuário
    nome_arquivo = list(uploaded.keys())[0]

    # Carrega os dados mapeados das Notas Fiscais
    with open(nome_arquivo, 'r', encoding='utf-8') as f:
        notas_carregadas = json.load(f)

    print("\n" + "="*60)
    print(f"Sucesso! {len(notas_carregadas)} notas carregadas para processamento.")
    print("="*60)

    # Executa a iteração direta de forma limpa
    for idx, nota in enumerate(notas_carregadas):
        num = nota.get('numeroNota')
        codigo = nota.get('codigoVerificacao')
        cnpj_prestador = nota.get('prestadorCnpj')
        razao_prestador = nota.get('prestadorNome')
        im_prestador = nota.get('prestadorIm')
        tomador = nota.get('tomadorNome')

        #print(f"[{idx+1}/{len(notas_carregadas)}] Preparando Nota: #{num} | Cód. Verificação: {codigo} | IM Prestador: {im_prestador}")

        # ===============================================================
        # COLOQUE AQUI A SUA FUNÇÃO ORIGINAL DO NOTEBOOK PARA BAIXAR O PDF
        # ===============================================================
        # Exemplo teórico de ativação do seu script original:
        # baixar_pdf_jundiai(
        #     numero_nota=num,
        #     codigo_verificacao=codigo,
        #     cnpj_prestador=cnpj_prestador,
        #     im_prestador=im_prestador
        # )
        # ===============================================================
else:
    print("Nenhum arquivo de parametrização enviado.")

In [ ]:
df_tmp = pd.DataFrame(notas_carregadas)
colunas = ['CNPJ_Prestador', 'Numero_NFSe', 'Codigo_Verificacao']
df_tmp = df_tmp[['prestadorCnpj', 'numeroNota', 'codigoVerificacao']]
df_tmp.columns = colunas
df_tabela = df_tmp
df_tabela

In [ ]:
import os
import time
import shutil
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from google.colab import files

# ================= CONFIGURAÇÕES =================
# Preencha com o CNPJ/CPF se quiser forçar o mesmo para todas as notas.
# Para usar o CNPJ dinâmico da planilha (df_tabela), deixe apenas as aspas vazias ("").
CNPJ_PRESTADOR_FORNECIDO = "17.658.262/0001-40"
URL_PORTAL = "https://jundiai.giss.com.br/portal/home#/verificacao-autenticidade-nfse"

# Cria pasta absoluta para salvar os PDFs
pasta_downloads = "/content/notas_pdf"
os.makedirs(pasta_downloads, exist_ok=True)

# Configurações do Chrome Invisível
chrome_options = webdriver.ChromeOptions()
chrome_options.binary_location = "/usr/bin/google-chrome" # <-- APONTA PARA O CHROME CORRETO
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')

prefs = {
    "download.default_directory": pasta_downloads,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True
}
chrome_options.add_experimental_option("prefs", prefs)

# Inicia o Navegador
driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 15)

# Força a permissão de downloads no modo invisível (Prevenção de erro do Colab)
driver.execute_cdp_cmd("Page.setDownloadBehavior", {
    "behavior": "allow",
    "downloadPath": pasta_downloads
})

print("Navegador iniciado com sucesso! Começando os downloads...\n")

# ================= LOOP DE DOWNLOADS =================
for index, row in df_tabela.iterrows():
    numero_nota = row['Numero_NFSe']
    codigo_verif = row['Codigo_Verificacao']

    # Lógica do CNPJ
    if CNPJ_PRESTADOR_FORNECIDO:
        cnpj_atual = re.sub(r'\D', '', str(CNPJ_PRESTADOR_FORNECIDO))
    else:
        cnpj_atual = row.get('CNPJ_Prestador', '')

    if not numero_nota or not codigo_verif or not cnpj_atual:
        print(f"Pulando linha... Faltam dados (Nota: {numero_nota}, Código: {codigo_verif}, CNPJ: {cnpj_atual})")
        continue

    print(f"Buscando Nota: {numero_nota} | Código: {codigo_verif} | CNPJ: {cnpj_atual}...")

    try:
        # Acessa a página
        driver.get(URL_PORTAL)

        # 0. Garante que o Radio Button "Por NFS-e" está selecionado
        radio_nfse = wait.until(EC.presence_of_element_located((By.ID, "porNfse")))
        driver.execute_script("arguments[0].click();", radio_nfse)

        # 1. Preenche o Número da Nota
        campo_numero = wait.until(EC.element_to_be_clickable((By.ID, "numeroNota")))
        campo_numero.clear()
        campo_numero.send_keys(str(numero_nota))

        # 2. Preenche o Código de Verificação
        campo_codigo = driver.find_element(By.ID, "codigoVerificador")
        campo_codigo.clear()
        campo_codigo.send_keys(str(codigo_verif))

        # 3. Preenche o CNPJ
        campo_cnpj = driver.find_element(By.ID, "documentoPrestador")
        campo_cnpj.clear()
        campo_cnpj.send_keys(str(cnpj_atual))

        # 4. Clica em Consultar
        # Usando XPath mais robusto baseado no HTML atual
        botao_consultar = driver.find_element(By.XPATH, "//button[@type='submit' and text()='Consultar']")
        driver.execute_script("arguments[0].click();", botao_consultar)

        # 5. Clica no ícone do PDF
        # Novo XPath baseado no HTML novo: <button title="Download PDF">
        xpath_pdf = "//button[@title='Download PDF']"
        botao_pdf = wait.until(EC.presence_of_element_located((By.XPATH, xpath_pdf)))

        # Clica via JavaScript para evitar erros de elemento sobreposto
        driver.execute_script("arguments[0].click();", botao_pdf)

        print(f"✓ PDF da nota {numero_nota} solicitado. Baixando...")
        time.sleep(5) # Espera 5 segundos para o servidor do governo entregar o download (Ajuste se necessário)

    except Exception as e:
        print(f"X Erro na nota {numero_nota}. Verifique manualmente. Motivo: Timeout ou não encontrada.")

driver.quit()

# ================= COMPACTAR E BAIXAR =================
print("\nDownloads concluídos! Preparando o arquivo ZIP...")

arquivo_zip = "/content/Notas_Fiscais_Baixadas"
shutil.make_archive(arquivo_zip, 'zip', pasta_downloads)

print("ZIP criado! Fazendo o download para o seu computador...")

files.download(arquivo_zip + ".zip")